<a href="https://colab.research.google.com/github/ankitta-singh/machinelearning/blob/main/05_Gradient_Boosting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import files
uploaded = files.upload()

Saving house-prices-advanced-regression-techniques.zip to house-prices-advanced-regression-techniques.zip


In [6]:
import zipfile
with zipfile.ZipFile("house-prices-advanced-regression-techniques.zip","r") as zip_ref:
  zip_ref.extractall("house_data")

In [12]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

In [13]:
df = pd.read_csv("house_data/train.csv")
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,...,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,...,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,...,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,...,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,...,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,...,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [14]:
x = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

In [15]:
cat_cols =x.select_dtypes(include="object").columns
num_cols = x.select_dtypes(exclude="object").columns
print("categorical columns:", len(cat_cols))
print("numerical column:", len(num_cols))

categorical columns: 43
numerical column: 37


In [16]:
missing = x.isnull().sum()
missing = missing[missing > 0]


In [20]:
for col in num_cols:
  x[col] = x[col].fillna(x[col].median())

for col in cat_cols:
  x[col] = x[col].fillna(x[col].mode()[0])

In [22]:
x = pd.get_dummies(x, drop_first=True)

In [23]:
print(x.isnull().sum().sum())

0


In [24]:
x_train, x_temp, y_train, y_temp = train_test_split(
    x,
    y,
    test_size=0.4,
    random_state=42)


In [25]:
x_val, x_test, y_val, y_test = train_test_split(
    x_temp,
    y_temp,
    test_size=0.5,
    random_state=42)

In [26]:
gb_model =GradientBoostingRegressor(
    random_state=42)

In [27]:
gb_model.fit(x_train, y_train)

GradientBoostingRegressor(random_state=42)

In [28]:
y_val_pred_gb = gb_model.predict(x_val)

In [29]:
y_val_pred_gb[:10]

array([134376.37828108, 411696.10123669, 307823.24995684, 131955.39700788,
       421538.63175048, 398295.30989312, 323430.49905788, 146380.63024273,
       235089.73136906, 130056.16817987])

In [31]:
print("Gradient Boosting Validation Results")
print("MSE:", mean_squared_error(y_val, y_val_pred_gb))
print("MAE:", mean_absolute_error(y_val, y_val_pred_gb))
print("R2:", r2_score(y_val, y_val_pred_gb))


Gradient Boosting Validation Results
MSE: 1483409872.2353182
MAE: 20147.02116161436
R2: 0.8424792289873628


In [46]:
param_grid_gb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

In [48]:
grid_search_gb = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid=param_grid_gb,
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [49]:
grid_search_gb.fit(x_train, y_train)

Fitting 5 folds for each of 32 candidates, totalling 160 fits


GridSearchCV(cv=5, estimator=GradientBoostingRegressor(random_state=42),
             n_jobs=-1,
             param_grid={'learning_rate': [0.05, 0.1], 'max_depth': [2, 3],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200]},
             verbose=2)

In [50]:
best_gb = grid_search_gb.best_estimator_

In [51]:
best_gb


GradientBoostingRegressor(learning_rate=0.05, n_estimators=200, random_state=42)

In [53]:
y_val_pred_tuned_gb = best_gb.predict(x_val)

In [54]:
y_val_pred_tuned_gb[:10]

array([132445.2813448 , 412237.3512498 , 313256.30101879, 133198.88487439,
       420359.1945524 , 368077.6908795 , 318243.89693251, 143883.9320607 ,
       245772.5923714 , 130971.06448313])

In [55]:
print("Tuned Gradient Boosting Validation Results")

print("MAE :", mean_absolute_error(y_val, y_val_pred_tuned_gb))

mse = mean_squared_error(y_val, y_val_pred_tuned_gb)
print("MSE :", mse)

print("RMSE:", np.sqrt(mse))

print("R2  :", r2_score(y_val, y_val_pred_tuned_gb))

Tuned Gradient Boosting Validation Results
MAE : 19036.63856223457
MSE : 1313944465.4879966
RMSE: 36248.37190120401
R2  : 0.8604744722646529


In [57]:
y_test_pred_tuned_gb = best_gb.predict(x_test)

In [58]:
print("Tuned Gradient Boosting Test Results")

print("MAE :", mean_absolute_error(y_test, y_test_pred_tuned_gb))

mse = mean_squared_error(y_test, y_test_pred_tuned_gb)
print("MSE :", mse)

print("RMSE:", np.sqrt(mse))

print("R2  :", r2_score(y_test, y_test_pred_tuned_gb))

Tuned Gradient Boosting Test Results
MAE : 14184.719242873949
MSE : 394106729.8296608
RMSE: 19852.12154480374
R2  : 0.919415286535979
